In [2]:
pip install pandas

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
     - -------------------------------------- 0.5/11.3 MB 3.3 MB/s eta 0:00:04
     --- ------------------------------------ 1.0/11.3 MB 2.8 MB/s eta 0:00:04
     ---- ----------------------------------- 1.3/11.3 MB 2.5 MB/s eta 0:00:05
     ------- -------------------------------- 2.1/11.3 MB 2.7 MB/s eta 0:00:04
     --------- ------------------------------ 2.6/11.3 MB 2.7 MB/s eta 0:00:04
     ----------- ---------------------------- 3.1/11.3 MB 2.7 MB/s eta 0:00:04
     ------------- -------------------------- 3.9/11.3 MB 2.9 MB/s eta 0:00:03
     ---------------- ----------------------- 4.7/11.3 MB 2.9 MB/s eta 0:00:03
     ------------------ --------------------- 5.2/11.3 MB 2.9 MB/s eta 0:00:03
     ---------------------- ----------------- 6.3/11.3 MB 3.0 MB/s eta 0:00:02
     ----------------------- ---------------- 6.6/11.3 MB 3.0 MB/s eta 0:00:0

In [3]:
import pandas as pd

file1 = "0312_2000_North0°_Overhang+Vertical(1).csv"
file2 = "0312_2000_North0°_Overhang+Vertical(2).csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

col = "out:Overhang_Length"

mask = df1[col] != df2[col]
diff_rows = pd.DataFrame({
    "Index": df1.loc[mask, "in:Index"],
    f"{col}_file1": df1.loc[mask, col],
    f"{col}_file2": df2.loc[mask, col],
    "差值": df1.loc[mask, col] - df2.loc[mask, col],
})

print(f"总行数: {len(df1)}")
print(f"不同的行数: {len(diff_rows)}")
print(f"相同的行数: {len(df1) - len(diff_rows)}")
print()
diff_rows

总行数: 2000
不同的行数: 255
相同的行数: 1745



,Index,out:Overhang_Length_file1,out:Overhang_Length_file2,差值
5,5,1.6,1.2,0.4
10,10,1.8,1.4,0.4
12,12,1.2,0.8,0.4
25,25,1.8,1.5,0.3
33,33,2.0,1.0,1.0
...,...,...,...,...
1940,1940,1.6,1.4,0.2
1949,1949,1.6,1.2,0.4
1954,1954,1.4,0.8,0.6
1961,1961,1.8,1.3,0.5


In [4]:
diff_indices = df1.loc[mask].index

diff_from_file1 = df1.loc[diff_indices]
diff_from_file2 = df2.loc[diff_indices]

diff_from_file1.to_csv("不同行_file1.csv", index=False)
diff_from_file2.to_csv("不同行_file2.csv", index=False)

print(f"已导出 {len(diff_from_file1)} 行到 不同行_file1.csv（来自文件1的所有列）")
print(f"已导出 {len(diff_from_file2)} 行到 不同行_file2.csv（来自文件2的所有列）")

已导出 255 行到 不同行_file1.csv（来自文件1的所有列）
已导出 255 行到 不同行_file2.csv（来自文件2的所有列）


In [5]:
df3 = pd.read_csv(r"../0312_2000_North0°_Overhang+Vertical.csv", header=None)

diff_index_list = diff_rows["Index"].tolist()

df3_diff = df3.loc[diff_index_list]

df3_diff.to_csv("不同行_原始文件.csv", index=False, header=False)

print(f"已从 0312_2000_North0°_Overhang+Vertical.csv 中提取 {len(df3_diff)} 行")
print(f"保存为: 不同行_原始文件.csv")

已从 0312_2000_North0°_Overhang+Vertical.csv 中提取 255 行
保存为: 不同行_原始文件.csv


In [6]:
import pandas as pd

recalc = pd.read_csv("255行重新计算.csv")
file2_diff = pd.read_csv("不同行_file2.csv")
target = pd.read_csv("0312_2000_North0°_Overhang+Vertical(2).csv")

original_indices = file2_diff["in:Index"].tolist()

print(f"重新计算的行数: {len(recalc)}")
print(f"不同行_file2 的行数: {len(file2_diff)}")
print(f"目标文件总行数: {len(target)}")
print(f"需要替换的 in:Index: {original_indices[:10]}... (共 {len(original_indices)} 个)")

assert len(recalc) == len(original_indices), "行数不匹配！"

recalc_cols = set(recalc.columns)
target_cols = set(target.columns)
print(f"\n重新计算文件独有列: {recalc_cols - target_cols}")
print(f"目标文件独有列: {target_cols - recalc_cols}")
print(f"共有列: {recalc_cols & target_cols}")

shared_cols = [c for c in recalc.columns if c in target.columns and c != "in:Index"]

for i, orig_idx in enumerate(original_indices):
    row_mask = target["in:Index"] == orig_idx
    assert row_mask.sum() == 1, f"in:Index={orig_idx} 在目标文件中找到 {row_mask.sum()} 行"
    target_row_idx = target.index[row_mask][0]
    for col in shared_cols:
        target.at[target_row_idx, col] = recalc.iloc[i][col]

output_file = "0312_2000_North0°_Overhang+Vertical(2)_replaced.csv"
target.to_csv(output_file, index=False)

print(f"\n已完成替换，结果保存为: {output_file}")
print(f"共替换 {len(original_indices)} 行, 每行替换 {len(shared_cols)} 个共有列")
print(f"替换的列: {shared_cols}")

重新计算的行数: 255
不同行_file2 的行数: 255
目标文件总行数: 2000
需要替换的 in:Index: [5, 10, 12, 25, 33, 35, 43, 65, 69, 81]... (共 255 个)

重新计算文件独有列: {'out:EUI(old)'}
目标文件独有列: {'out:Rank', 'out:Hypervolume', 'out:PMV'}
共有列: {'out:EUI', 'out:Overhang_Movement_Distance', 'out:Overhang_Angle', 'out:sGA', 'out:WWR_Skylight_Vertical_Shading_Divided', 'out:H_Skylight', 'out:Distance_A_Corridor', 'out:Vertiacl_Shading_Top_Left_Movement_Distance', 'out:WWR_Corridor', 'out:Overhang_Length', 'out:Degree', 'out:Vertical_Shading_Width', 'out:Vertical_Shading_Number', 'out:Distance_B_Corridor', 'out:Vertical_Shading_Length', 'out:W_Skylight_Vertical_Shading_Divided', 'out:Distance_A_Skylight', 'out:W_Skylight', 'out:Distance_B_Skylight', 'in:Index', 'out:Vertiacl_Shading_Top_Right_Movement_Distance', 'out:W_Corridor', 'out:sDA', 'out:SH_Skylight', 'out:UDI', 'out:SH_Corridor', 'out:SVF', 'out:WWR_Skylight', 'out:H_Corridor'}


C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_51000\293729522.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '30.994543' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  target.at[target_row_idx, col] = recalc.iloc[i][col]



已完成替换，结果保存为: 0312_2000_North0°_Overhang+Vertical(2)_replaced.csv
共替换 255 行, 每行替换 28 个共有列
替换的列: ['out:WWR_Skylight', 'out:WWR_Skylight_Vertical_Shading_Divided', 'out:H_Skylight', 'out:W_Skylight', 'out:SH_Skylight', 'out:Distance_A_Skylight', 'out:Distance_B_Skylight', 'out:WWR_Corridor', 'out:H_Corridor', 'out:W_Corridor', 'out:SH_Corridor', 'out:Distance_A_Corridor', 'out:Distance_B_Corridor', 'out:Degree', 'out:Overhang_Angle', 'out:Overhang_Length', 'out:Overhang_Movement_Distance', 'out:Vertical_Shading_Number', 'out:Vertical_Shading_Width', 'out:Vertical_Shading_Length', 'out:W_Skylight_Vertical_Shading_Divided', 'out:Vertiacl_Shading_Top_Left_Movement_Distance', 'out:Vertiacl_Shading_Top_Right_Movement_Distance', 'out:EUI', 'out:sDA', 'out:sGA', 'out:UDI', 'out:SVF']


In [ ]:
import pandas as pd

recalc = pd.read_csv("255行重新计算.csv")
file1_diff = pd.read_csv("不同行_file1.csv")
file2_diff = pd.read_csv("不同行_file2.csv")
target = pd.read_csv("0312_2000_North0°_Overhang+Vertical(1).csv")

original_indices = file1_diff["in:Index"].tolist()

print(f"重新计算的行数: {len(recalc)}")
print(f"不同行_file1 的行数: {len(file1_diff)}")
print(f"目标文件总行数: {len(target)}")
print(f"需要替换的 in:Index (前10个): {original_indices[:10]}... (共 {len(original_indices)} 个)")

# 验证: out:WWR_Skylight ~ out:Vertiacl_Shading_Top_Right_Movement_Distance
# 是否在 255行重新计算.csv 和 不同行_file2.csv 之间一致
input_cols = [c for c in recalc.columns[1:24]]  # out:WWR_Skylight ~ out:Vertiacl_Shading_Top_Right_Movement_Distance
mismatch_input = 0
for col in input_cols:
    for i in range(len(recalc)):
        if abs(float(recalc.iloc[i][col]) - float(file2_diff.iloc[i][col])) > 1e-6:
            mismatch_input += 1
if mismatch_input == 0:
    print(f"\n验证通过: out:WWR_Skylight ~ out:Vertiacl_Shading_Top_Right_Movement_Distance")
    print(f"  在 255行重新计算.csv 和 不同行_file2.csv 之间完全一致! (共检查 {len(input_cols)}列 x {len(recalc)}行 = {len(input_cols)*len(recalc)} 个值)")
else:
    print(f"\n警告: 有 {mismatch_input} 个值不一致!")

# 获取 recalc 和 target 之间的共有列（排除 in:Index）
shared_cols = [c for c in recalc.columns if c in target.columns and c != "in:Index"]
print(f"\n共有列 ({len(shared_cols)} 列): {shared_cols}")

# 替换所有共有列
for col in shared_cols:
    if target[col].dtype != recalc[col].dtype:
        try:
            target[col] = target[col].astype(float)
        except:
            pass

for i, orig_idx in enumerate(original_indices):
    row_mask = target["in:Index"] == orig_idx
    assert row_mask.sum() == 1, f"in:Index={orig_idx} 在目标文件中找到 {row_mask.sum()} 行"
    target_row_idx = target.index[row_mask][0]
    for col in shared_cols:
        target.at[target_row_idx, col] = recalc.iloc[i][col]

# 将替换行的 out:sDA 和 out:sGA 从 0~1 转为百分比（×100）
mask = target["in:Index"].isin(original_indices)
target.loc[mask, "out:sDA"] = target.loc[mask, "out:sDA"] * 100
target.loc[mask, "out:sGA"] = target.loc[mask, "out:sGA"] * 100

output_file = "0312_2000_North0°_Overhang+Vertical(1)_replaced.csv"
target.to_csv(output_file, index=False)

print(f"\n已完成替换，结果保存为: {output_file}")
print(f"共替换 {len(original_indices)} 行, 每行替换 {len(shared_cols)} 列")
print("out:sDA 和 out:sGA 已转换为百分比形式 (×100)")

# 抽样验证
result = pd.read_csv(output_file)
print("\n抽样验证 (前5个替换行):")
print(f"{'in:Index':>10}  {'sDA':>12}  {'sGA':>12}  {'EUI':>12}")
for i in range(5):
    idx = original_indices[i]
    t = result.loc[result["in:Index"] == idx].iloc[0]
    print(f"{idx:>10}  {t['out:sDA']:>12.4f}  {t['out:sGA']:>12.4f}  {t['out:EUI']:>12.3f}")